In [ ]:
import starfile
import os
from pathlib import Path
import pandas as pd
import numpy as np
import einops
import mrcfile
import typer
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch

from torch_image_lerp import insert_into_image_2d
from torch_affine_utils import homogenise_coordinates
from torch_affine_utils.transforms_2d import R,T,S
from torch_grid_utils.coordinate_grid import coordinate_grid

In [2]:
from typing import Literal

import einops
import torch
from torch_affine_utils import homogenise_coordinates
from torch_grid_utils import coordinate_grid
from torch_image_interpolation import sample_image_2d


def affine_transform_image_2d(
    image: torch.Tensor,
    output_shape: tuple[int, int],
    matrices: torch.Tensor,
    interpolation: Literal['nearest', 'bilinear', 'bicubic']
) -> torch.Tensor:
    # grab image dimensions
    h, w = output_shape

    # generate grid of pixel coordinates
    grid = coordinate_grid(image_shape=(h, w), device=image.device)

    # apply matrix to coordinates
    grid = homogenise_coordinates(grid)  # (h, w, yxw)
    grid = einops.rearrange(grid, 'h w yxw -> h w yxw 1')
    grid = matrices @ grid
    grid = grid[..., :2, 0]  # dehomogenise coordinates: (..., h, w, yxw, 1) -> (..., h, w, yx)

    # sample image at transformed positions
    result = sample_image_2d(image, coordinates=grid, interpolation=interpolation)
    return result

In [27]:
def suppress_membrane():
    try:
        tomo_name = list(Path.cwd().glob("*.rawtlt"))[0].stem
    except IndexError:
        typer.echo("No rawtlt file found")
        raise typer.Exit()
    
    tiltseries_data = starfile.read(f"{tomo_name}_relion_tiltseries.star")
    membrain_filename = f"membrain/{tomo_name}_full_rec_BP_3DCTF_BIN4_ddw_MemBrain_seg_v10_alpha.ckpt_segmented.mrc"
    
    from subprocess import Popen, PIPE
    for idx, row in tiltseries_data.iterrows():
        process = Popen(["xyzproj", "-inp", membrain_filename, 
                                    "-o", f"membrain/proj{idx:02}.mrc",
                                    "-ax","Y",
                                    "-ta",str(row["rlnTomoYTilt"]),
                                    "-mo","2"], stdout=PIPE, stderr=PIPE)
        process.communicate()
        with mrcfile.open(f"membrain/proj{idx:02}.mrc", mode='r+') as mrc:
            membrain_proj=torch.tensor(mrc.data[0])
        with mrcfile.open(Path(row["rlnMicrographName"]).parent / (Path(row["rlnMicrographName"]).stem + "_aunpremoved.mrc")) as mrc_file:
            img_tensor = torch.from_numpy(mrc_file.data)
        Tc = T(-torch.tensor(membrain_proj.shape) / 2)
        Rz = R(row["rlnTomoZRot"],yx=True)
        Ts = T(torch.tensor(row[["rlnTomoYShiftAngst", "rlnTomoXShiftAngst"]]/10.0))
        Sc = S([4.0,4.0])
        Tfc = T(torch.tensor(img_tensor.shape) / 2)
        M = Tfc @ Sc @ Ts @ Rz @ Tc
        transformed_image = affine_transform_image_2d(
            image=membrain_proj,
            output_shape=img_tensor.shape,
            matrices=torch.linalg.inv(M),
            interpolation='nearest'
        )
        ms_image = img_tensor - (img_tensor - img_tensor.mean()) * (transformed_image / transformed_image.max())
        mrcfile.write(Path(row["rlnMicrographName"]).parent / (Path(row["rlnMicrographName"]).stem + "_aunpremoved_membranesupressed.mrc"), ms_image.numpy(), overwrite=True)

        

In [ ]:
#os.chdir('/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/20231026_HippAu_14/best_alignment')
#suppress_membrane()

/tmp/ipykernel_1960056/1994709214.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/tmp/ipykernel_1960056/1994709214.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/tmp/ipykernel_1960056/1994709214.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/tmp/ipykernel_1960056/1994709214.py:25: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labe

In [ ]:
with open("/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/tomo_for_sta.txt","r") as f:
    tomos_for_sta = f.readlines()

#tomos_for_sta = ["20240209_waffleHip_165"]
for tomo in tomos_for_sta:
    tomo = tomo.strip()
    print(tomo)
    os.chdir(f'/goliath/processing/Gouaux/CJS/BestTomo/ProcessingCJS/tomograms/15F1_tomograms/TOP_TOMOS/{tomo}/best_alignment/')
    suppress_membrane()